# Demand Forecasting System

## Notebook 01 - Exploratory Data Analysis (EDA)

### Objective

The goal of this notebook is to understand the Rossmann Store Sales dataset before any preprocessing or model development.

Tasks:
- Load the datasets
- Inspect structure and data types
- Understand each feature
- Identify missing values
- Explore the target variable (Sales)
- Understand temporal trends
- Prepare for feature engineering

In [ ]:
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', None) # so that our column cells are not truncated
pd.set_option('display.max_rows', 10) # here 10 means if the dataset is larger than 10 rows, pandas would truncate it to 10 rows 
train = pd.read_csv("../data/train.csv")
store = pd.read_csv("../data/store.csv")

#print the first 5 rows of the dataset
train.head()
store.head()

In [ ]:
train.info()
store.info()

## Observations after train.info() and store.info()

1. The training dataset contains 1,017,209 daily sales records from 1,115 stores.
2. The training dataset has no missing values.
3. The Date column is currently stored as a string and will later be converted to datetime.
4. The StateHoliday column has mixed data types and requires further inspection.
5. The store dataset contains store-level metadata such as competition distance, store type, and assortment.
6. Missing values exist in CompetitionDistance, Promo2SinceYear, and PromoInterval. These may represent business conditions rather than data quality issues.

In [ ]:
1+1

In [ ]:
#every statistical details are given about the dataset
train.describe()
store.describe()

## Observations after `describe()`

### Train Dataset

1. The average daily sales per store are approximately **5774**.
2. Some stores recorded **zero sales**, which requires further investigation.
3. Stores receive an average of **633 customers per day**.
4. Around **83%** of the observations correspond to stores that were open.
5. Promotions are active on approximately **38%** of the recorded days.
6. School holidays account for around **18%** of the observations.

### Store Dataset

1. The median competition distance (2325 m) is much lower than the mean (5405 m), suggesting a right-skewed distribution.
2. Approximately half of the stores participate in the Promo2 program.
3. Some competition-related fields contain unusual values (e.g., year = 1900) that should be investigated during EDA.

## observations after unique()

| Column        | Type        | Why?                                             |
| ------------- | ----------- | ------------------------------------------------ |
| Store         | Identifier  | Unique store ID, not a measurable quantity       |
| DayOfWeek     | Categorical | Labels for weekdays                              |
| Date          | Date/Time   | Contains temporal information to derive features |
| Sales         | Numerical   | Continuous target variable                       |
| Customers     | Numerical   | Count of customers                               |
| Open          | Binary      | 0 or 1                                           |
| Promo         | Binary      | 0 or 1                                           |
| StateHoliday  | Categorical | Multiple holiday categories (`0`, `a`, `b`, `c`) |
| SchoolHoliday | Binary      | 0 or 1                                           |

| Column                    | Type                                            |
| ------------------------- | ----------------------------------------------- |
| Store                     | Identifier                                      |
| StoreType                 | Categorical                                     |
| Assortment                | Categorical                                     |
| CompetitionDistance       | Numerical                                       |
| CompetitionOpenSinceMonth | Numerical (will be used in feature engineering) |
| CompetitionOpenSinceYear  | Numerical (will be transformed later)           |
| Promo2                    | Binary                                          |
| Promo2SinceWeek           | Numerical                                       |
| Promo2SinceYear           | Numerical                                       |
| PromoInterval             | Categorical                                     |


In [ ]:
#check which columns have null values or not
train.isnull()
store.isnull()

store["Promo2"].value_counts()
store.groupby("Promo2")["Promo2SinceYear"].count()
store.groupby("Promo2")["PromoInterval"].count()

## Observations after Missing Value Analysis

1. The training dataset contains no missing values.

2. Missing values exist only in the `store` dataset.

3. Missing values related to `Promo2SinceWeek`, `Promo2SinceYear`, and `PromoInterval` occur only for stores where `Promo2 = 0`.

4. These missing values do **not** indicate poor data quality. They represent stores that never participated in the Promo2 program.

5. Therefore, these missing values should be handled using business logic rather than arbitrary imputation.

### Hypothesis

We suspect that the missing values in the promotion-related columns occur because some stores never participated in the Promo2 program.

To verify this hypothesis, we compare the availability of promotion-related features for stores with and without Promo2.

In [ ]:
print("Duplicate rows in train dataset :", train.duplicated().sum())
print("Duplicate rows in store dataset :", store.duplicated().sum())

In [ ]:
df = pd.merge(train, store, on="Store", how="left")
df.shape
df.head()
df.info()

### Why do we merge the datasets?

The `train` dataset contains daily sales transactions but lacks store-specific information such as store type, assortment, and competition details.

The `store` dataset contains these attributes for each store.

By merging both datasets using the common key `Store`, every daily sales record is enriched with the corresponding store information. `Store` acts as a foreign key here.

The number of rows remain the same that is 1017209, number of columns might have increased due to some additional information we get from the store dataset.

here "left" means "left outer join" which indicates Keep all rows from the left dataframe (train). If a match is found in the right dataframe (store), append its columns. If no match is found, fill the new columns with NaN (missing values). 

In [ ]:
df["Date"].dtype
df["Date"] = pd.to_datetime(df["Date"]) # here we convert the string dtype data to actual timestamos dtype dates
df["Date"].dtype

In [ ]:
df["Date"].dt.month
df["Date"].dt.weekday
df["Date"].dt.day
df["Date"].dt.year


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,5))

sns.histplot(df["Sales"], bins=100, kde=True)

plt.title("Distribution of Sales")
plt.xlabel("Sales")
plt.ylabel("Frequency")

plt.show()

In [ ]:
df[df["Sales"] == 0]["Open"].value_counts()

In [ ]:
df[(df["Sales"] == 0) & (df["Open"] == 1)]

## Observations

1. The Sales distribution is highly right-skewed.
2. Most daily sales values lie within a moderate range, while a small number of observations represent exceptionally high sales.
3. A large spike at zero sales is observed, indicating many days with no recorded sales.
4. Several high-value outliers are present, likely corresponding to genuine business events rather than data errors.

## Business Interpretation

- Typical sales are concentrated in a relatively narrow range.
- High-sales days are rare but important for forecasting.
- Zero-sales days require further investigation before model development.

## Key Takeaways

- The target variable is not normally distributed.
- Zero-sales observations must be investigated.
- Outliers should be examined before deciding whether to retain or transform them.

## Additional Observation

Among all zero-sales observations:

- 99.97% correspond to stores that were closed.
- Only 54 observations recorded zero sales despite the store being open.
- Holidays do not explain these exceptions.
- Further investigation of the 54 observations where `Open = 1` and `Sales = 0` revealed that all   of them had `Customers = 0`.

This indicates that the stores were operational but did not receive any customers, making zero sales a reasonable business outcome rather than a data quality issue.

# 9. Sales Trend Over Time

## Business Question

Demand forecasting is fundamentally a time series problem.

Rather than analyzing sales at the individual store level, we first examine the company's total daily sales over time.

This helps us identify:

- Overall trend
- Seasonality
- Sudden spikes and drops
- Business cycles

These insights are essential before selecting an appropriate forecasting model.

In [ ]:
#we did this since we have sales info about one store one day, and suppose a company have 1115 stores
#so will it plot the whole 1115 points or just one point for one store
#so we have to calculate the total sales of all the stores on that day, referrring it as a single point
# (total company sales that day)

daily_sales = (
    df.groupby("Date")["Sales"]
      .sum()
      .reset_index()
)

#we do this reset index thing to convert the pandas series to normal data frame
#( basically an index (0/1) column is added for plotting convinience )

In [ ]:
plt.figure(figsize=(16,6))

plt.plot(
    daily_sales["Date"],
    daily_sales["Sales"],
    linewidth=1.5
)

plt.title("Total Daily Sales Over Time", fontsize=16)

plt.xlabel("Date")

plt.ylabel("Total Sales")

plt.grid(alpha=0.3)

plt.show()

In [ ]:
monthly_sales = (
    df.groupby(pd.Grouper(key="Date", freq="ME"))["Sales"]
      .sum()
      .reset_index()
)

plt.figure(figsize=(15,6))

plt.plot(
    monthly_sales["Date"],
    monthly_sales["Sales"],
    marker="o",
    linewidth=2
)

plt.title("Monthly Total Sales")
plt.xlabel("Month")
plt.ylabel("Sales")

plt.grid(alpha=0.3)

plt.show()

In [ ]:
#this gives on which date the sales were the highest

monthly_sales.sort_values("Sales", ascending=False).head(5)


## Observations

1. Monthly sales fluctuate over time but do not exhibit a strong long-term increasing or decreasing trend.
2. The data shows recurring monthly variations, indicating that time influences sales.
3. January 2014 records the highest monthly sales during the observation period.
4. A noticeable decline in monthly sales is observed during the second half of 2014.
5. Monthly aggregation reveals business patterns that were difficult to identify in the daily sales plot.

## Business Interpretation

- The business appears relatively stable over the observed period.
- Demand varies across months, suggesting temporal effects that should be incorporated into forecasting models.
- Certain months experience exceptionally high or low sales, warranting further investigation into promotions, holidays, or operational factors.

## Key Takeaways

- Monthly aggregation provides a clearer understanding of business performance than daily sales.
- Time is an important predictor for forecasting future demand.
- Additional analyses on promotions and holidays are needed to explain the observed fluctuations.

### Additional Monthly Trend Insights

To support the visual interpretation, the months with the highest total sales were identified.

The highest monthly sales were recorded in:

- December 2013
- July 2015
- July 2013
- June 2015
- March 2015

December 2013 recorded the highest sales during the observation period. July appears multiple times among the highest-performing months, suggesting a potential seasonal effect that can be explored further.

### Key takeaway
"Prepare higher inventory for historically strong months like December and potentially July."


# 10. Sales by Day of Week

## Business Question

Customer shopping behavior often varies across different days of the week.

Understanding this pattern helps businesses optimize:

- Inventory planning
- Workforce allocation
- Promotional campaigns

This analysis investigates whether average sales differ by weekday.

In [ ]:
day_sales = (
    df.groupby("DayOfWeek")["Sales"]
      .mean()
      .reset_index()
)

day_names = {
    1: "Mon",
    2: "Tue",
    3: "Wed",
    4: "Thu",
    5: "Fri",
    6: "Sat",
    7: "Sun"
}

day_sales["DayName"] = day_sales["DayOfWeek"].map(day_names)

plt.figure(figsize=(10,5))

plt.bar(day_sales["DayName"], day_sales["Sales"])

plt.title("Average Sales by Day of Week", fontsize=16)

plt.xlabel("Day")

plt.ylabel("Average Sales")

plt.grid(axis="y", alpha=0.3)

plt.show()

## Observations

1. Monday records the highest average sales among all weekdays.
2. Sunday records extremely low average sales.
3. Average sales from Monday to Saturday remain within a relatively narrow range.
4. Sunday behaves very differently from the rest of the week.

## Business Interpretation

- Weekly shopping behaviour affects sales.
- Extremely low Sunday sales are primarily explained by widespread store closures.
- Day of the week is an important temporal feature for demand forecasting.

## Key Takeaways

- Weekly patterns exist in the data.
- `DayOfWeek` should be retained as an important forecasting feature.
- Sunday should be interpreted in the context of store operating hours rather than customer demand alone.

# 11. Impact of Promotions on Sales

## Business Question

Promotional campaigns are one of the most influential drivers of retail demand.

This analysis investigates whether stores experience higher sales on promotional days compared to non-promotional days.

Understanding this relationship helps businesses:

- Plan inventory
- Allocate marketing budgets
- Estimate future demand during promotional campaigns

In [ ]:
plt.figure(figsize=(8,6))

sns.boxplot(
    x="Promo",
    y="Sales",
    data=df
)

plt.title("Sales Distribution on Promotion vs Non-Promotion Days",
          fontsize=15)

plt.xlabel("Promotion")

plt.ylabel("Sales")

plt.xticks([0,1],["No Promotion","Promotion"])

plt.grid(alpha=0.3)

plt.show()

## Observations

1. Promotional days have a significantly higher median sales value than non-promotional days.
2. The entire interquartile range (IQR) is shifted upward during promotions, indicating consistently higher sales.
3. Both promotional and non-promotional days contain high-value outliers. So promotion is not only the factor influencing sales.
4. Sales variability is greater during promotions, suggesting that the impact of promotions differs across stores.

## Business Interpretation

- Promotions have a positive impact on retail sales.
- The increase is consistent across the central portion of the distribution, not driven solely by a few extreme observations.
- Different stores respond differently to promotional campaigns, highlighting the importance of store-specific characteristics. This captures the variance in sales due to promotion factors.

## Key Takeaways

- Promotion is an important predictor of future sales.
- Promotional campaigns should be included as a key feature in the forecasting model.
- Store-level differences should also be considered because promotional impact is not uniform.

## Why Boxplot?

- Different stores respond differently to promotional campaigns, highlighting the importance of store-specific characteristics. This captures the variance in sales due to promotion factors.

- This variance cannot be captured by the linear bar graph, for that we need boxplot. Also,
A bar chart can onl ysay Promotional Sales are higher but the box plot tells us by how much they are higher.

# 12. Impact of Holidays on Sales

## Business Question

Retail demand is often influenced by holidays.

Different types of state holidays may affect customer purchasing behaviour differently.

This analysis compares the **average sales** across different holiday categories to understand their impact on retail demand.

In [ ]:
holiday_sales

what the problem arise here is, we have two type of dtype signifying no holidays, one is integer(the first one) anothre one is string(the second one), this is thereason we were getting an error or it was showing that stateholidays column is "object" meaning it has mixed dtypes. so we have to clean the data now.

In [ ]:
df["StateHoliday"] = df["StateHoliday"].astype(str)

In [ ]:
print(df["StateHoliday"].unique())

In [ ]:
holiday_sales = (
    df.groupby("StateHoliday")["Sales"]
      .mean()
      .reset_index()
)

In [ ]:
holiday_sales["Holiday"] = holiday_sales["StateHoliday"].map(holiday_names)

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    holiday_sales["Holiday"],
    holiday_sales["Sales"]
)

plt.title("Average Sales by State Holiday")

plt.xlabel("Holiday Type")

plt.ylabel("Average Sales")

plt.grid(axis="y", alpha=0.3)

plt.show()

## Observations

1. Sales are highest on non-holiday days.
2. All three holiday categories show significantly lower average sales.
3. Christmas records the lowest average sales.
4. State holidays have a strong negative association with daily sales.

## Business Interpretation

- State holidays are associated with substantially lower retail sales.
- This reduction is likely influenced by store closures and reduced business operations rather than customer demand alone.
- Holiday information should therefore be incorporated into demand forecasting models.

## Key Takeaways

- StateHoliday is an important forecasting feature.
- Holiday periods require different inventory and staffing strategies compared to regular business days.

In [ ]:
#school holidays
school_sales = (
    df.groupby("SchoolHoliday")["Sales"]
      .mean()
      .reset_index()
)

plt.figure(figsize=(7,5))

plt.bar(
    ["No School Holiday", "School Holiday"],
    school_sales["Sales"]
)

plt.title("Average Sales by School Holiday")

plt.ylabel("Average Sales")

plt.grid(axis="y", alpha=0.3)

plt.show()

## Observations

1. Average sales are higher during school holidays.
2. Unlike state holidays, school holidays do not reduce sales.
3. The difference suggests that school holidays positively influence customer shopping behaviour.

## Business Interpretation

- School holidays appear to encourage higher retail activity.
- Families may spend more time shopping during school breaks.
- SchoolHoliday should be retained as an important feature for demand forecasting.

## Key Takeaways

- Not all holidays affect sales in the same way.
- State holidays and school holidays capture different aspects of customer behaviour and business operations.

# 13. Impact of Store Type on Sales

## Business Question

Rossmann operates multiple types of stores, each catering to different customer segments and product assortments.

This analysis aims to answer the following question:

> **Do different store types generate different levels of average sales?**

Understanding this relationship helps businesses:

- Identify high-performing store formats.
- Plan future store expansion.
- Allocate inventory based on store characteristics.
- Build more accurate demand forecasting models by incorporating store-specific effects.

In [ ]:
storetype_sales = (
    df.groupby("StoreType")["Sales"]
      .mean()
      .reset_index()
)

plt.figure(figsize=(8,5))

plt.bar(
    storetype_sales["StoreType"],
    storetype_sales["Sales"]
)

plt.title("Average Sales by Store Type")

plt.xlabel("Store Type")

plt.ylabel("Average Sales")

plt.grid(axis="y", alpha=0.3)

plt.show()

## Observations

1. Store Type B records the highest average sales among all store formats.
2. Store Types A, C and D exhibit relatively similar average sales.
3. The difference between Store Type B and the remaining store types is substantial.

## Business Interpretation

- Store format significantly influences sales performance.
- Store Type B appears to attract higher customer demand or operates under conditions that generate greater sales.
- StoreType should be included as an important feature in the forecasting model.

## Key Takeaways

- Different store formats contribute differently to sales.
- Inventory planning and demand forecasting should account for store type.

why is store B so different?

# 14. Impact of Assortment on Sales

## Business Question

Rossmann stores offer different levels of product assortment (product variety).

This analysis aims to answer the following question:

> **Does offering a wider range of products lead to higher average sales?**

Understanding this relationship helps businesses:

- Determine whether increasing product variety boosts customer purchases.
- Decide the optimal assortment strategy for different store locations.
- Improve inventory planning by identifying which assortment types generate the highest demand.
- Include product assortment as a meaningful feature in the demand forecasting model.

In [ ]:
assortment_sales = (
    df.groupby("Assortment")["Sales"]
      .mean()
      .reset_index()
)

plt.figure(figsize=(8,5))

plt.bar(
    assortment_sales["Assortment"],
    assortment_sales["Sales"]
)

plt.title("Average Sales by Assortment")

plt.xlabel("Assortment")

plt.ylabel("Average Sales")

plt.grid(axis="y", alpha=0.3)

plt.show()

## Observations

1. Assortment Type B records the highest average sales.
2. Assortment Type C performs moderately better than Assortment Type A.
3. Average sales vary across different assortment types.

## Business Interpretation

- Product assortment is associated with sales performance.
- Stores offering certain assortment types consistently achieve higher average sales.
- Assortment should be considered an important feature in the demand forecasting model.

## Key Takeaways

- Product assortment influences sales patterns.
- Inventory planning should consider the assortment strategy adopted by each store.
- Further analysis is needed to separate the effect of assortment from other store characteristics.

Assortment means the variety of the products offered from the company.

# 15. Impact of Competition Distance on Sales

## Business Question

Competitor stores located near a Rossmann outlet may influence customer purchasing behaviour.

This analysis aims to answer the following question:

> **Does the distance to the nearest competitor affect average sales?**

Understanding this relationship helps businesses:

- Evaluate the impact of nearby competitors.
- Identify locations with competitive pressure.
- Support future store expansion decisions.
- Include competition-related information in the demand forecasting model.

In [ ]:
df["CompetitionBin"] = pd.cut(
    df["CompetitionDistance"],
    bins=5
)

competition_sales = (
    df.groupby("CompetitionBin", observed=True)["Sales"]
      .mean()
      .reset_index()
)

plt.figure(figsize=(10,5))

plt.bar(
    competition_sales["CompetitionBin"].astype(str),
    competition_sales["Sales"]
)

plt.xticks(rotation=20)

plt.title("Average Sales by Competition Distance")

plt.xlabel("Competition Distance")

plt.ylabel("Average Sales")

plt.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.show()

## Observations

1. Average sales do not exhibit a clear monotonic relationship with competition distance.
2. Stores in the largest competition-distance category record the highest average sales.
3. The first four distance categories have relatively similar average sales.This suggests that within normal competition distances, other factors (such as promotions, store type, assortment, holidays, etc.) may have a larger impact on sales.
4. Competition distance alone does not fully explain variations in sales.

## Business Interpretation

- Competition distance appears to influence sales, but its effect is weaker than features such as promotions or store type.
- Other store-specific characteristics are likely interacting with competition distance.
- CompetitionDistance should be retained as a useful feature for the forecasting model.

## Key Takeaways

- Competition distance provides additional business context.
- Sales are influenced by multiple interacting factors rather than a single variable.

# 16. Correlation Analysis

## Business Question

Individual feature analysis helps us understand one variable at a time.

However, real-world demand is influenced by multiple variables simultaneously.

This analysis aims to answer:

> **How strongly are the numerical features related to each other and to Sales?**

Understanding these relationships helps:

- Identify the most influential numerical predictors.
- Detect redundant features.
- Gain insights before building machine learning models.

In [ ]:
plt.figure(figsize=(10,8))

correlation = df.select_dtypes(include=np.number).corr()

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")

plt.show()

## Observations

1. Customers exhibit the strongest positive correlation with Sales (0.89).
2. Store Open status also shows a strong positive correlation with Sales (0.68).
3. Promotions have a moderate positive correlation with Sales (0.45).
4. DayOfWeek has a moderate negative correlation with Sales (-0.46), largely due to lower Sunday sales.
5. CompetitionDistance and SchoolHoliday show weak linear correlations with Sales.

## Business Interpretation

- Customer traffic is the strongest indicator of sales, although it cannot be used for future forecasting because it is unknown beforehand.
- Promotions and store operational status significantly influence sales.
- Competition distance appears to have only a weak direct linear relationship with sales.

## Key Takeaways

- Promotions and temporal features are important forecasting variables.
- Correlation should be interpreted alongside business knowledge and previous analyses.
- Weak correlation does not necessarily imply a feature is unimportant, especially for tree-based models.

# Executive Summary of EDA

The exploratory data analysis revealed several important business insights:

1. Sales are highly right-skewed, with a few exceptionally high-sales days.
2. Nearly all zero-sales observations occur when stores are closed.
3. Sales exhibit clear weekly and monthly temporal patterns.
4. Promotions consistently increase average sales.
5. State holidays reduce sales, whereas school holidays are associated with higher sales.
6. Store Type B achieves substantially higher average sales than the other store formats.
7. Assortment Type B records the highest average sales among all assortment categories.
8. Competition distance shows only a weak direct relationship with sales.
9. Customers, Open status, and Promotions exhibit the strongest numerical relationships with Sales.
10. The EDA confirms that demand is influenced by multiple interacting business factors rather than a single variable.